In [ ]:
import os
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# SSL 환경 변수 초기화 (PostgreSQL 에러 방지)
for env_var in ['REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE']:
    if env_var in os.environ:
        del os.environ[env_var]

def get_twosome_final_boss():
    chrome_options = Options()
    # 진행 상황을 눈으로 확인하기 위해 창을 띄웁니다.
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    url = "https://mo.twosome.co.kr/so/storeList.do"
    
    try:
        driver.get(url)
        wait = WebDriverWait(driver, 15)

        # 1. [전체매장] 버튼이 로드될 때까지 기다리고 확실히 클릭
        print("🖱️ '전체매장' 탭으로 전환 중...")
        all_btn_selector = 'a[data="all"]'
        all_store_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, all_btn_selector)))
        
        # 버튼을 클릭하고 해당 탭이 활성화(보통 class에 on이나 active가 붙음)될 때까지 대기
        driver.execute_script("arguments[0].click();", all_store_btn)
        time.sleep(3) # 탭 전환 후 첫 데이터 로딩 시간 확보

        # 2. 무한 스크롤 (1,700개 목표)
        print("📜 전국 매장 스캔을 시작합니다. 잠시만 기다려 주세요...")
        
        last_count = 0
        retry_limit = 5 # 데이터가 안 늘어날 때 최대 5번까지 재시도 (더 끈질기게)
        no_change_count = 0
        
        while True:
            # 현재 로드된 매장 개수 파악
            current_items = driver.find_elements(By.CSS_SELECTOR, "#infiniteScroll > li[data]")
            current_count = len(current_items)
            
            # 실시간 진행 상황 출력
            print(f"🔄 현재 수집된 매장: {current_count}개...", end="\r")

            if current_count > last_count:
                # 데이터가 늘어났다면 카운트 초기화
                last_count = current_count
                no_change_count = 0
            else:
                # 데이터가 안 늘어난다면
                no_change_count += 1
                
            # 5번 연속으로 숫자가 안 늘어나면 정말 끝인 것으로 판단
            if no_change_count >= retry_limit:
                # 만약 숫자가 너무 적은데(예: 300개 미만) 멈췄다면 강제로 스크롤 한 번 더 시도
                if current_count < 300:
                    driver.execute_script("window.scrollTo(0, 0);") # 위로 갔다가
                    time.sleep(1)
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") # 다시 아래로
                    no_change_count = 0 # 기회 한 번 더
                    continue
                break
            
            # 아래로 끝까지 스크롤
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2.0) # 서버 응답 대기 시간

        print(f"\n✅ 스캔 종료! 최종 {len(current_items)}개 매장을 찾았습니다.")

        # 3. BeautifulSoup 파싱
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        store_list = soup.select("#infiniteScroll > li[data]")

        results = []
        for store in store_list:
            store_id = store.get('data', '결측치')
            
            # 매장명 추출
            dt = store.find('dt', class_='cf')
            store_nm = dt.find('strong').get_text(strip=True) if dt and dt.find('strong') else "결측치"
            
            # 주소 추출
            dd = store.find('dd', id='addr')
            store_addr = dd.get_text(strip=True) if dd else "결측치"

            results.append({
                "지점ID": store_id,
                "매점명": store_nm,
                "매점주소": store_addr
            })

        return pd.DataFrame(results)

    except Exception as e:
        print(f"\n❌ 오류 발생: {e}")
        return pd.DataFrame()
    finally:
        driver.quit()

# 실행 및 저장
df_twosome = get_twosome_final_boss()

if not df_twosome.empty:
    # 중복 제거 (혹시 모를 중복 수집 방지)
    df_twosome = df_twosome.drop_duplicates(subset=['지점ID'])
    
    print("\n" + "="*50)
    print(f"📊 최종 수집 결과: {len(df_twosome)}개")
    print("="*50)
    print(df_twosome.head(5))
    
    df_twosome.to_csv("twosome_place.csv", index=False, encoding="utf-8-sig")
    print("\n💾 'twosome_place.csv' 파일로 저장되었습니다.")

🖱️ '전체매장' 탭으로 전환 중...
📜 전국 매장 스캔을 시작합니다. 잠시만 기다려 주세요...
🔄 현재 수집된 매장: 1731개...
✅ 스캔 종료! 최종 1731개 매장을 찾았습니다.

📊 최종 수집 결과: 1731개
      지점ID          매점명                                               매점주소
0  1903894      전농동사거리점                        서울특별시 동대문구 사가정로 97 (전농동) 1층
1  1903871  파주오두산통일전망대점                  경기도 파주시 탄현면 필승로 369 (오두산통일전망대) 4층
2  1903869    인천영종한빛타워점        인천광역시 중구 쪽빛하늘로 72 (운남동) 한빛타워 106, 107, 108호
3  1903788         홍제역점  서울특별시 서대문구 통일로 455 (홍제동, 경인주유소) 서울시 서대문구  홍제동 ...
4  1903851      대구지산광장점               대구광역시 수성구 용학로 297-7 (지산동) 1,2층 (지산동)

💾 'twosome_final_data.csv' 파일로 저장되었습니다.


In [9]:
import os
import requests
import pandas as pd
import time
import random
from bs4 import BeautifulSoup
from tqdm import tqdm
import urllib3

# 1. SSL 인증서 에러 방지
for env_var in ['REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE']:
    if env_var in os.environ:
        del os.environ[env_var]

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def retry_failed_connections():
    file_path = "twosome_place.csv"
    
    if not os.path.exists(file_path):
        print(f"❌ '{file_path}' 파일이 없습니다.")
        return

    # CSV 로드
    df = pd.read_csv(file_path)
    
    # [핵심] 전화번호가 "연결실패"인 행들만 필터링해서 리스트 생성
    # 아직 컬럼이 없거나 빈 값인 경우도 포함하도록 조건 설정
    target_mask = (df['전화번호'] == "연결실패") | (df['전화번호'].isna())
    targets = df[target_mask]
    
    if len(targets) == 0:
        print("✅ 모든 매장의 전화번호가 이미 수집되었습니다!")
        return

    print(f"🕵️ 총 {len(targets)}개의 '연결실패' 매장에 대해 재수집을 시작합니다.")
    print("💡 팁: 실행 전 휴대폰 핫스팟 등으로 IP를 바꾸면 더 잘 됩니다.")

    base_url = "https://mo.twosome.co.kr/so/storeDetail.do?storeCd={}"
    user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    ]

    # 업데이트된 내용을 즉시 반영하기 위해 iterrows 사용
    for i in tqdm(targets.index, desc="재시도 중"):
        store_id = df.at[i, '지점ID']
        url = base_url.format(store_id)
        
        headers = {"User-Agent": random.choice(user_agents)}
        
        try:
            # 타임아웃을 15초로 넉넉하게 잡습니다.
            res = requests.get(url, headers=headers, verify=False, timeout=15)
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.text, 'html.parser')
                phone_tag = soup.select_one('a[href^="tel:"]')
                phone = phone_tag.get_text(strip=True) if phone_tag else "결측치"
                
                # 성공 시 데이터프레임의 해당 행 업데이트
                df.at[i, '전화번호'] = phone
                
            elif res.status_code == 403 or res.status_code == 429:
                print(f"\n🚫 또 차단되었습니다. IP를 변경하고 잠시 후에 다시 실행하세요.")
                break # 차단 시 루프 중단
            else:
                # 500 에러 등 다른 서버 에러인 경우
                df.at[i, '전화번호'] = f"에러({res.status_code})"
                
        except Exception:
            # 여전히 연결이 안 되면 그대로 "연결실패" 유지
            df.at[i, '전화번호'] = "연결실패"
            
        # 사람처럼 보이게 랜덤 휴식 (중요!)
        time.sleep(random.uniform(0.8, 1.8))
        
        # 30개마다 중간 저장 (불안하니까요!)
        if i % 30 == 0:
            df.to_csv(file_path, index=False, encoding="utf-8-sig")

    # 최종 결과 저장 (기존 파일 덮어쓰기)
    df.to_csv(file_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 재수집 작업 완료! '{file_path}'를 확인하세요.")

# 실행
retry_failed_connections()

🕵️ 총 1441개의 '연결실패' 매장에 대해 재수집을 시작합니다.
💡 팁: 실행 전 휴대폰 핫스팟 등으로 IP를 바꾸면 더 잘 됩니다.


재시도 중: 100%|██████████| 1441/1441 [35:52<00:00,  1.49s/it]


✅ 재수집 작업 완료! 'twosome_place.csv'를 확인하세요.


In [1]:
import os
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 1. SSL 인증서 에러 방지 (지긋지긋한 PostgreSQL 경로 에러 해결)
for env_var in ['REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE']:
    if env_var in os.environ:
        del os.environ[env_var]

def get_phone_with_selenium():
    file_path = "twosome_place.csv"
    if not os.path.exists(file_path):
        print("❌ 'twosome_place.csv' 파일이 없습니다.")
        return

    df = pd.read_csv(file_path)
    
    # '연결실패'이거나 아직 번호가 없는 행만 골라내기
    target_indices = df[df['전화번호'].isin(['연결실패', '연결실패 ', None]) | df['전화번호'].isna()].index
    
    if len(target_indices) == 0:
        print("✅ 모든 전화번호가 수집되었습니다!")
        return

    print(f"🕵️ 총 {len(target_indices)}개의 '연결실패' 매장을 Selenium으로 재시도합니다.")

    # 브라우저 설정 (최대한 사람처럼 보이게)
    chrome_options = Options()
    # 차단을 피하기 위해 창을 띄우는 것을 권장하지만, 너무 번거로우면 headless를 켜세요.
    # chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # 자동화 감지 우회
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    # 봇 감지 우회를 위해 웹드라이버 속성 제거
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })

    base_url = "https://mo.twosome.co.kr/so/storeDetail.do?storeCd={}"

    try:
        # 진행률을 위해 enumerate 사용
        for i, idx in enumerate(target_indices):
            store_id = df.at[idx, '지점ID']
            url = base_url.format(store_id)
            
            try:
                driver.get(url)
                # 페이지 로딩 대기 (td 태그가 나타날 때까지 최대 10초)
                wait = WebDriverWait(driver, 10)
                
                # 전화번호 링크(tel:)가 포함된 a 태그 찾기
                phone_el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="tel:"]')))
                phone_number = phone_el.text.strip()
                
                df.at[idx, '전화번호'] = phone_number
                print(f"[{i+1}/{len(target_indices)}] {df.at[idx, '매점명']}: {phone_number}")

            except Exception:
                print(f"[{i+1}/{len(target_indices)}] {df.at[idx, '매점명']}: 여전히 실패")
                df.at[idx, '전화번호'] = "연결실패"

            # 🛑 중요: 사람처럼 보이게 랜덤하게 쉽니다.
            time.sleep(random.uniform(1.5, 3.5))

            # 30개 수집할 때마다 브라우저를 껐다 켜거나 1분간 휴식 (차단 방지 핵심)
            if (i + 1) % 30 == 0:
                print("\n💤 서버 의심을 피하기 위해 1분간 휴식합니다...")
                time.sleep(60)
                # 100개마다 중간 저장
                df.to_csv(file_path, index=False, encoding="utf-8-sig")

    finally:
        df.to_csv(file_path, index=False, encoding="utf-8-sig")
        driver.quit()
        print(f"\n✅ 작업 완료! '{file_path}'가 업데이트되었습니다.")

# 실행
get_phone_with_selenium()

🕵️ 총 816개의 '연결실패' 매장을 Selenium으로 재시도합니다.
[1/816] 대전대서문점: 042-283-5633
[2/816] 구미옥계점: 054-475-0860
[3/816] 신내SKV1센터점: 02-6252-5321
[4/816] 서일대점: 02-2208-1030
[5/816] 부천중동위브점: 032-620-5812
[6/816] 군산지곡호수공원점: 063-463-8700
[7/816] 고양삼송미디어시티점: 02-2039-9122
[8/816] 경희대국제캠퍼스점: 031-203-2316
[9/816] 도봉산입구점: 02-3492-7778
[10/816] 용인처인구청점: 031-334-0770
[11/816] 전주반월DT점: 063-212-6600
[12/816] 대전원신흥점: 042-825-5778
[13/816] 대전목원대점: 042-826-1607
[14/816] 송파잠실H타워점: 02-417-4360
[15/816] 서판교점: 031-705-0778
[16/816] 별내용암천점: 031-571-1222
[17/816] 제주오라우정점: 064-747-3637
[18/816] 대구반월당점: 053-253-3356
[19/816] 양평양수점: 031-775-0722
[20/816] 청계천광교점: 02-2135-5622
[21/816] 진주세란병원점: 187-7-0112
[22/816] 광주두암타운점: 062-266-3388
[23/816] 제일제당사옥점: 02-2039-2188
[24/816] 기흥AK몰점: 031-693-6087
[25/816] 제천고암케이원점: 043-645-2222
[26/816] 경산대로점: 053-792-9295
[27/816] 대구월배아이파크점: 053-285-3424
[28/816] 서판교운중타운점: 031-8039-7970
[29/816] 목포석현점: 061-801-7589
[30/816] 대전판암역점: 042-273-0888

💤 서버 의심을 피하기 위해 1분간 휴식합니다...
[31/816] 일산덕이점: 031

In [1]:
import pandas as pd
from sqlalchemy import create_engine

# 1. CSV 파일 로드 (파일명: twosome_place.csv)
file_path = 'twosome_place.csv'
df = pd.read_csv(file_path, encoding='utf-8-sig')

# [데이터 정제] 혹시 모를 결측치 처리
df['전화번호'] = df['전화번호'].fillna('결측치')

# 2. MySQL 연결 설정
# 형식: mysql+pymysql://사용자명:비밀번호@호스트:포트/데이터베이스명
# 본인의 MySQL 정보로 수정하세요! (예: root / 1234)
user = "root"
password = "1234"
host = "localhost"
port = "3306"
db_name = "coffee_store"

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}?charset=utf8mb4")

# 3. DB에 데이터 밀어넣기
try:
    # if_exists='append': 이미 테이블이 있으므로 데이터를 추가함
    # index=False: 판다스 인덱스는 저장하지 않음
    df.to_sql(name='twosome', con=engine, if_exists='append', index=False)
    print(f"🎉 성공! '{file_path}'의 데이터가 'coffee_store.twosome' 테이블로 이동되었습니다.")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

🎉 성공! 'twosome_place.csv'의 데이터가 'coffee_store.twosome' 테이블로 이동되었습니다.
